# Batch Cryptojacking Detection

Run the trained 2-gram Random Forest over the saved test sparse matrices, explain each prediction with SHAP, and request evidence-grounded LLM analysis only for samples predicted as malware. LLM prompts contain RF/SHAP features and occurrence metadata, not full function disassembly.

Per-sample LLM results are saved under `llm_results/`. Per-sample pipeline reports, an aggregate CSV, and a run summary are saved under `batch_detection_output/`.

In [1]:
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "ml" / "feature_datasets").is_dir()
        and (candidate / "graph_aware_classifier").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root from the notebook's working directory."
    )

TEST_DATA_DIR = PROJECT_ROOT / "ml" / "feature_datasets" / "2gram" / "test"
GRAPH_DATASET_DIR = PROJECT_ROOT / "ml" / "dataset"
OCCURRENCES_DIR = PROJECT_ROOT / "graph_aware_classifier" / "occurences"
LLM_RESULTS_DIR = PROJECT_ROOT / "graph_aware_classifier" / "llm_results"
BATCH_OUTPUT_DIR = PROJECT_ROOT / "graph_aware_classifier" / "batch_detection_output"

NGRAM_SIZE = 2
SHAP_TOP_FEATURES = 20
CLASS_NAMES = {0: "Benign", 1: "Cryptojacking"}

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Repository: {PROJECT_ROOT}")
print(f"Sparse test data: {TEST_DATA_DIR}")
print(f"Per-sample LLM results: {LLM_RESULTS_DIR}")
print(f"Batch reports: {BATCH_OUTPUT_DIR}")

Repository: /home/azureuser/Graph-Aware-Cryptojacking-Classifier
Sparse test data: /home/azureuser/Graph-Aware-Cryptojacking-Classifier/ml/feature_datasets/2gram/test
Per-sample LLM results: /home/azureuser/Graph-Aware-Cryptojacking-Classifier/graph_aware_classifier/llm_results
Batch reports: /home/azureuser/Graph-Aware-Cryptojacking-Classifier/graph_aware_classifier/batch_detection_output


In [2]:
import json
import re

import numpy as np
import shap
from scipy.sparse import load_npz

import graph_aware_classifier.sample_explainer as sample_explainer
from graph_aware_classifier.extract_sample_graph_ngrams import (
    extract_sample_graph_ngrams,
)

# ---------------------------------------------------------------------------
# Load trained model and vocabulary
# ---------------------------------------------------------------------------
rf, vocabulary = sample_explainer._load_model_and_vocabulary(NGRAM_SIZE)

if len(vocabulary) != rf.n_features_in_:
    raise ValueError(
        f"Vocabulary has {len(vocabulary)} features but the RF expects "
        f"{rf.n_features_in_}"
    )

positive_class_indices = np.flatnonzero(rf.classes_ == 1)
if positive_class_indices.size != 1:
    raise ValueError(
        f"Expected exactly one class-1 label, found {rf.classes_}"
    )

positive_class_index = int(positive_class_indices[0])

# Build feature names required by the SHAP explanations.
feature_names = [
    " ".join(vocabulary.get_entry(feature_id).ngram)
    for feature_id in range(len(vocabulary))
]

# The vocabulary is no longer needed after feature_names is built.
del vocabulary

# Keep one SHAP explainer for the entire run rather than rebuilding it
# for every sample.
shap_explainer = shap.TreeExplainer(rf)


def resolve_source_json(row_metadata):
    """Resolve a test row to its original graph JSON using row metadata."""
    source_name = Path(row_metadata["source_name"]).name
    metadata_path = row_metadata.get("source_path")

    if metadata_path:
        candidate = Path(metadata_path)
        if candidate.is_file():
            return candidate.resolve()

    class_name = row_metadata["class_name"]
    candidate = GRAPH_DATASET_DIR / class_name / source_name

    if not candidate.is_file():
        raise FileNotFoundError(
            f"Source graph JSON not found for {source_name!r}; "
            f"tried metadata path {metadata_path!r} and {candidate}"
        )

    return candidate.resolve()


def safe_sample_name(source_name):
    name = Path(source_name).name

    if name.lower().endswith(".json"):
        name = name[:-5]

    return re.sub(r"[^\w.-]", "_", name)


if not TEST_DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Test sparse-matrix directory not found: {TEST_DATA_DIR}"
    )

batch_files = sorted(
    TEST_DATA_DIR.glob("X_batch_*.npz")
)

if not batch_files:
    raise FileNotFoundError(
        f"No X_batch_*.npz matrices found in {TEST_DATA_DIR}"
    )

# ---------------------------------------------------------------------------
# Build and validate batch manifest
# ---------------------------------------------------------------------------
# Only file paths and metadata needed to locate each batch are retained here.
batch_manifest = []
seen_report_names = set()
total_test_rows = 0

for matrix_path in batch_files:
    try:
        batch_number = int(
            matrix_path.stem.rsplit("_", 1)[1]
        )
    except (IndexError, ValueError) as exc:
        raise ValueError(
            f"Unexpected sparse matrix filename: {matrix_path.name}"
        ) from exc

    labels_path = (
        TEST_DATA_DIR
        / f"y_batch_{batch_number:03d}.npy"
    )

    metadata_path = (
        TEST_DATA_DIR
        / f"row_metadata_batch_{batch_number:03d}.json"
    )

    if not labels_path.is_file():
        raise FileNotFoundError(
            f"Labels are missing for {matrix_path.name}: {labels_path}"
        )

    if not metadata_path.is_file():
        raise FileNotFoundError(
            f"Row metadata is missing for {matrix_path.name}: "
            f"{metadata_path}"
        )

    matrix = load_npz(matrix_path)
    labels = np.load(
        labels_path,
        allow_pickle=False,
    )

    with metadata_path.open(
        "r",
        encoding="utf-8",
    ) as metadata_file:
        row_metadata = json.load(metadata_file)

    if matrix.shape[1] != rf.n_features_in_:
        raise ValueError(
            f"{matrix_path.name} has {matrix.shape[1]} features; "
            f"the model expects {rf.n_features_in_}"
        )

    if (
        matrix.shape[0] != len(labels)
        or matrix.shape[0] != len(row_metadata)
    ):
        raise ValueError(
            f"Batch {batch_number:03d} row mismatch: "
            f"matrix={matrix.shape[0]}, "
            f"labels={len(labels)}, "
            f"metadata={len(row_metadata)}"
        )

    for row_index, metadata in enumerate(row_metadata):
        actual_label = int(labels[row_index])

        expected_class = {
            0: "benign",
            1: "cryptojacking",
        }.get(actual_label)

        if expected_class is None:
            raise ValueError(
                f"Unexpected label {actual_label} in {labels_path}"
            )

        if int(metadata["label"]) != actual_label:
            raise ValueError(
                f"Label mismatch in batch {batch_number:03d}, "
                f"row {row_index}: "
                f"y={actual_label}, "
                f"metadata={metadata['label']}"
            )

        if metadata.get("class_name") != expected_class:
            raise ValueError(
                f"Class mismatch in batch {batch_number:03d}, "
                f"row {row_index}: "
                f"expected {expected_class!r}, "
                f"got {metadata.get('class_name')!r}"
            )

        if int(
            metadata.get(
                "batch_number",
                batch_number,
            )
        ) != batch_number:
            raise ValueError(
                f"Batch number mismatch in {metadata_path}, "
                f"row {row_index}"
            )

        if int(
            metadata.get(
                "row_index_in_batch",
                row_index,
            )
        ) != row_index:
            raise ValueError(
                f"Row index mismatch in {metadata_path}, "
                f"row {row_index}"
            )

        source_path = resolve_source_json(metadata)

        report_name = (
            f"{safe_sample_name(metadata['source_name'])}"
            "_analysis.json"
        )

        if report_name in seen_report_names:
            raise ValueError(
                f"More than one test row maps to "
                f"report filename {report_name}"
            )

        seen_report_names.add(report_name)

    batch_manifest.append(
        {
            "batch_number": batch_number,
            "matrix_path": matrix_path,
            "labels_path": labels_path,
            "metadata_path": metadata_path,
        }
    )

    total_test_rows += matrix.shape[0]

    # Release this validation batch immediately.
    # The manifest retains only file paths, so the data itself is no longer
    # needed until the processing cell loads the batch again.
    del matrix
    del labels
    del row_metadata

print(
    f"Validated {len(batch_manifest)} sparse batches "
    f"and {total_test_rows} test rows."
)

/home/azureuser/Graph-Aware-Cryptojacking-Classifier/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/azureuser/Graph-Aware-Cryptojacking-Classifier/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/azureuser/Graph-Aware-Cryptojacking-Classifier/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.9.1. This might lead to break

Validated 9 sparse batches and 881 test rows.


In [3]:
import csv
from datetime import datetime, timezone

from graph_aware_classifier.llm_analyzer import LLMAnalyzer

OCCURRENCES_DIR.mkdir(parents=True, exist_ok=True)
LLM_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

analyzer = None
processed_count = 0
rf_malware_count = 0
llm_completed_count = 0
llm_failed_count = 0
reused_report_count = 0

for batch_info in batch_manifest:
    batch_number = batch_info["batch_number"]

    matrix = load_npz(batch_info["matrix_path"]).tocsr()
    labels = np.load(batch_info["labels_path"], allow_pickle=False)

    with batch_info["metadata_path"].open(
        "r",
        encoding="utf-8",
    ) as metadata_file:
        row_metadata = json.load(metadata_file)

    print(
        f"Processing batch {batch_number:03d}: "
        f"{matrix.shape[0]} samples"
    )

    for row_index in range(matrix.shape[0]):
        metadata = row_metadata[row_index]
        source_path = resolve_source_json(metadata)

        source_name = Path(metadata["source_name"]).name
        binary_name = (
            source_name[:-5]
            if source_name.lower().endswith(".json")
            else source_name
        )
        safe_name = safe_sample_name(source_name)
        actual_label = int(labels[row_index])

        report_path = BATCH_OUTPUT_DIR / f"{safe_name}_analysis.json"

        existing_report = None

        if report_path.is_file():
            try:
                with report_path.open(
                    "r",
                    encoding="utf-8",
                ) as report_file:
                    existing_report = json.load(report_file)
            except (OSError, json.JSONDecodeError):
                existing_report = None

        if isinstance(existing_report, dict):
            existing_sample = existing_report.get("sample")
            existing_prediction = existing_report.get("random_forest")
            existing_shap = existing_report.get("shap")
            existing_llm = existing_report.get("llm")

            report_is_reusable = (
                isinstance(existing_sample, dict)
                and isinstance(existing_prediction, dict)
                and isinstance(existing_shap, dict)
                and isinstance(existing_llm, dict)
                and existing_sample.get("source_name") == source_name
                and existing_sample.get("source_path")
                == source_path.relative_to(PROJECT_ROOT).as_posix()
                and existing_sample.get("batch_number") == batch_number
                and existing_sample.get("row_index_in_batch") == row_index
                and existing_sample.get("actual_label") == actual_label
                and existing_prediction.get("ngram_size") == NGRAM_SIZE
                and existing_shap.get("status") == "completed"
                and existing_llm.get("status") in {"completed", "skipped"}
            )

            if report_is_reusable:
                existing_predicted_class = existing_prediction.get(
                    "predicted_class"
                )
                existing_llm_status = existing_llm.get("status")

                processed_count += 1
                reused_report_count += 1

                if existing_predicted_class == 1:
                    rf_malware_count += 1

                if existing_llm_status == "completed":
                    llm_completed_count += 1
                elif existing_llm_status == "failed":
                    llm_failed_count += 1

                print(
                    f"  [{processed_count}/{total_test_rows}] "
                    f"{binary_name}: reused existing report -> "
                    f"{report_path.name}"
                )

                del existing_report
                del existing_sample
                del existing_prediction
                del existing_shap
                del existing_llm

                continue

        sparse_row = matrix.getrow(row_index)

        predicted_class = int(rf.predict(sparse_row)[0])
        malware_probability = float(
            rf.predict_proba(sparse_row)[0, positive_class_index]
        )

        report = {
            "sample": {
                "binary_name": binary_name,
                "source_name": source_name,
                "source_path": source_path.relative_to(PROJECT_ROOT).as_posix(),
                "split": metadata.get("split", "test"),
                "batch_number": batch_number,
                "row_index_in_batch": row_index,
                "actual_label": actual_label,
                "actual_class": metadata["class_name"],
            },
            "random_forest": {
                "ngram_size": NGRAM_SIZE,
                "malware_probability": malware_probability,
                "predicted_class": predicted_class,
                "predicted_class_name": CLASS_NAMES[predicted_class],
            },
            "shap": {
                "top_feature_limit": SHAP_TOP_FEATURES,
                "top_features": [],
                "status": "pending",
            },
            "llm": {
                "status": "skipped",
                "reason": "The RF prediction was benign.",
                "token_usage": {
                    "input_tokens": 0,
                    "output_tokens": 0,
                    "source": "not_called",
                },
            },
        }

        # Initialize these so cleanup is safe even if SHAP fails.
        top_features = []
        dense_row = None
        raw_explanation = None
        explanation = None
        llm_result = None

        try:
            dense_row = sparse_row.toarray().astype(
                np.float32,
                copy=False,
            )

            raw_explanation = shap_explainer(dense_row)

            explanation = shap.Explanation(
                values=raw_explanation.values,
                base_values=raw_explanation.base_values,
                data=dense_row,
                feature_names=feature_names,
            )

            top_features = sample_explainer.ranked_feature_contributions(
                explanation,
                class_index=positive_class_index,
                limit=SHAP_TOP_FEATURES,
            )

            report["shap"]["top_features"] = top_features
            report["shap"]["status"] = "completed"

        except Exception as exc:
            report["shap"]["status"] = "failed"
            report["shap"]["error"] = (
                f"{type(exc).__name__}: {exc}"
            )
            report["llm"] = {
                "status": "skipped",
                "reason": (
                    "SHAP analysis failed; no evidence was sent to the LLM."
                ),
                "token_usage": {
                    "input_tokens": 0,
                    "output_tokens": 0,
                    "source": "not_called",
                },
            }

        finally:
            # Release large SHAP-related objects immediately.
            dense_row = None
            raw_explanation = None
            explanation = None

        if (
            predicted_class == 1
            and report["shap"]["status"] == "completed"
        ):
            occurrence_path = (
                OCCURRENCES_DIR
                / f"{safe_name}_vocabulary.json"
            )

            try:
                occurrence_report = extract_sample_graph_ngrams(
                    source_path,
                    n=NGRAM_SIZE,
                )

                with occurrence_path.open(
                    "w",
                    encoding="utf-8",
                ) as occurrence_file:
                    json.dump(
                        occurrence_report,
                        occurrence_file,
                        indent=2,
                        ensure_ascii=False,
                    )

                del occurrence_report

                if analyzer is None:
                    analyzer = LLMAnalyzer(
                        results_dir=LLM_RESULTS_DIR
                    )

                llm_result = analyzer.analyze_sample(
                    binary_name=binary_name,
                    rf_prediction=predicted_class,
                    rf_probability=malware_probability,
                    top_features=top_features,
                    occurrences_path=occurrence_path,
                )

                usage = llm_result.get(
                    "token_usage",
                    {
                        "input_tokens": None,
                        "output_tokens": None,
                        "source": "unavailable",
                    },
                )

                report["llm"] = {
                    "status": "completed",
                    "analysis": llm_result["analysis"],
                    "token_usage": usage,
                    "result_path": (
                        LLM_RESULTS_DIR
                        / f"{safe_name}_llm_result.json"
                    )
                    .relative_to(PROJECT_ROOT)
                    .as_posix(),
                }

            except Exception as exc:
                report["llm"] = {
                    "status": "failed",
                    "error": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                    "token_usage": {
                        "input_tokens": None,
                        "output_tokens": None,
                        "source": "unavailable",
                    },
                }

        # Write the complete per-sample report to disk.
        with report_path.open(
            "w",
            encoding="utf-8",
        ) as report_file:
            json.dump(
                report,
                report_file,
                indent=2,
                ensure_ascii=False,
                allow_nan=False,
            )

        # Update counters after the final report has been written.
        processed_count += 1

        if predicted_class == 1:
            rf_malware_count += 1

        if report["llm"]["status"] == "completed":
            llm_completed_count += 1
        elif report["llm"]["status"] == "failed":
            llm_failed_count += 1

        print(
            f"  [{processed_count}/{total_test_rows}] "
            f"{binary_name}: "
            f"{CLASS_NAMES[predicted_class]} "
            f"(p={malware_probability:.4f}), "
            f"LLM {report['llm']['status']} -> "
            f"{report_path.name}"
        )

        # The report is safely on disk now, so release the
        # sample-specific objects before processing the next row.
        del llm_result
        del report
        del top_features
        del sparse_row

    # Release batch-level data before loading the next batch.
    del matrix
    del labels
    del row_metadata

print(
    f"Finished {processed_count} test rows. "
    f"Reused existing reports: {reused_report_count}; "
    f"RF malware predictions: {rf_malware_count}; "
    f"LLM completed: {llm_completed_count}; "
    f"LLM failed: {llm_failed_count}."
)

Processing batch 000: 100 samples
  [1/881] x86__64__lsb__unix-system-v__clang-3.8.0__O3__no-obf__unstripped__coreutils-8.30__expand: reused existing report -> x86__64__lsb__unix-system-v__clang-3.8.0__O3__no-obf__unstripped__coreutils-8.30__expand_analysis.json
  [2/881] mobsync.exe: reused existing report -> mobsync.exe_analysis.json
  [3/881] 00bba2efa9e9bb020828cde686e51ea3a39de47dabf65f197b18e957257f69c0.exe: reused existing report -> 00bba2efa9e9bb020828cde686e51ea3a39de47dabf65f197b18e957257f69c0.exe_analysis.json
  [4/881] comrepl.exe: reused existing report -> comrepl.exe_analysis.json
  [5/881] ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0__O0__no-obf__unstripped__coreutils-8__copy_1_.30__hostid: reused existing report -> ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0__O0__no-obf__unstripped__coreutils-8__copy_1_.30__hostid_analysis.json
  [6/881] SlideToShutDown.exe: reused existing report -> SlideToShutDown.exe_analysis.json
  [7/881] setx.exe: reus

2026-10-06 05:52:16,072 [INFO] graph_aware_classifier.llm_analyzer — Starting LLM analysis for df336ee7abd31a141d76107f73603a71e3859fc142f04333f1c0b1b8633ea800.exe …
2026-10-06 05:52:54,522 [INFO] httpx2 — HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-10-06 05:52:54,637 [INFO] graph_aware_classifier.llm_analyzer — LLM result saved → /home/azureuser/Graph-Aware-Cryptojacking-Classifier/graph_aware_classifier/llm_results/df336ee7abd31a141d76107f73603a71e3859fc142f04333f1c0b1b8633ea800.exe_llm_result.json


  [70/881] df336ee7abd31a141d76107f73603a71e3859fc142f04333f1c0b1b8633ea800.exe: Cryptojacking (p=1.0000), LLM completed -> df336ee7abd31a141d76107f73603a71e3859fc142f04333f1c0b1b8633ea800.exe_analysis.json
  [71/881] x86__64__lsb__unix-system-v__clang-3.8.0__O1__no-obf__unstripped__coreutils-8.30__ln: reused existing report -> x86__64__lsb__unix-system-v__clang-3.8.0__O1__no-obf__unstripped__coreutils-8.30__ln_analysis.json
  [72/881] ba2d1265f9355b41a1e6a5571ae72ac5550cb045993b0fa9dc399264f0861845.exe: reused existing report -> ba2d1265f9355b41a1e6a5571ae72ac5550cb045993b0fa9dc399264f0861845.exe_analysis.json


2026-10-06 05:53:21,936 [INFO] graph_aware_classifier.llm_analyzer — Starting LLM analysis for 18905f26968cceda8c5e765a88212c80c279f716a4c80620596a2779ca3425d8.elf …
2026-10-06 05:53:47,736 [INFO] httpx2 — HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-10-06 05:53:47,763 [INFO] graph_aware_classifier.llm_analyzer — LLM result saved → /home/azureuser/Graph-Aware-Cryptojacking-Classifier/graph_aware_classifier/llm_results/18905f26968cceda8c5e765a88212c80c279f716a4c80620596a2779ca3425d8.elf_llm_result.json


  [73/881] 18905f26968cceda8c5e765a88212c80c279f716a4c80620596a2779ca3425d8.elf: Cryptojacking (p=1.0000), LLM completed -> 18905f26968cceda8c5e765a88212c80c279f716a4c80620596a2779ca3425d8.elf_analysis.json


...

  [869/881] 08be3260b1569540aa2ea8c55d4b1f099e735fced9917e5e058b06a16c674fe1.elf: Cryptojacking (p=0.9100), LLM completed -> 08be3260b1569540aa2ea8c55d4b1f099e735fced9917e5e058b06a16c674fe1.elf_analysis.json
  [870/881] wfascim.dll: Benign (p=0.0000), LLM skipped -> wfascim.dll_analysis.json


2026-10-06 08:31:56,982 [INFO] graph_aware_classifier.llm_analyzer — Starting LLM analysis for 6ffafaa40005d4dff0436ac9b18cce45d99d6b106e840c0cad22fe08e31d2f5f.elf …
2026-10-06 08:32:21,583 [INFO] httpx2 — HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-10-06 08:32:21,601 [INFO] graph_aware_classifier.llm_analyzer — LLM result saved → /home/azureuser/Graph-Aware-Cryptojacking-Classifier/graph_aware_classifier/llm_results/6ffafaa40005d4dff0436ac9b18cce45d99d6b106e840c0cad22fe08e31d2f5f.elf_llm_result.json


  [871/881] 6ffafaa40005d4dff0436ac9b18cce45d99d6b106e840c0cad22fe08e31d2f5f.elf: Cryptojacking (p=1.0000), LLM completed -> 6ffafaa40005d4dff0436ac9b18cce45d99d6b106e840c0cad22fe08e31d2f5f.elf_analysis.json
  [872/881] ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0__O0__no-obf__unstripped__coreutils-8__copy_1_.30__realpath: Benign (p=0.0000), LLM skipped -> ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0__O0__no-obf__unstripped__coreutils-8__copy_1_.30__realpath_analysis.json
  [873/881] ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0__O1__no-obf__unstripped__coreutils-8.30__printf: Benign (p=0.0000), LLM skipped -> ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0__O1__no-obf__unstripped__coreutils-8.30__printf_analysis.json
  [874/881] ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0__O0__no-obf__unstripped__coreutils-8__copy_1_.30__shuf: Benign (p=0.1300), LLM skipped -> ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0_

2026-10-06 08:32:23,244 [INFO] graph_aware_classifier.llm_analyzer — Starting LLM analysis for 63fb1bb6af66fbe29823c5ac577034207888c3bf23218c96777ec63bf7499afd.exe …
2026-10-06 08:32:45,310 [INFO] httpx2 — HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-10-06 08:32:45,313 [INFO] graph_aware_classifier.llm_analyzer — LLM result saved → /home/azureuser/Graph-Aware-Cryptojacking-Classifier/graph_aware_classifier/llm_results/63fb1bb6af66fbe29823c5ac577034207888c3bf23218c96777ec63bf7499afd.exe_llm_result.json


  [877/881] 63fb1bb6af66fbe29823c5ac577034207888c3bf23218c96777ec63bf7499afd.exe: Cryptojacking (p=0.9850), LLM completed -> 63fb1bb6af66fbe29823c5ac577034207888c3bf23218c96777ec63bf7499afd.exe_analysis.json
  [878/881] x86__64__lsb__unix-system-v__clang-3.8.0__O0__no-obf__unstripped__coreutils-8.30__mktemp: Benign (p=0.0000), LLM skipped -> x86__64__lsb__unix-system-v__clang-3.8.0__O0__no-obf__unstripped__coreutils-8.30__mktemp_analysis.json
  [879/881] ChtIME.exe: Benign (p=0.0000), LLM skipped -> ChtIME.exe_analysis.json
  [880/881] x86__64__lsb__unix-system-v__clang-3.8.0__O0__no-obf__unstripped__util-linux-2.33__sulogin: Benign (p=0.0100), LLM skipped -> x86__64__lsb__unix-system-v__clang-3.8.0__O0__no-obf__unstripped__util-linux-2.33__sulogin_analysis.json
  [881/881] SensorDataService.exe: Benign (p=0.0000), LLM skipped -> SensorDataService.exe_analysis.json
Finished 881 test rows. Reused existing reports: 71; RF malware predictions: 261; LLM completed: 261; LLM failed: 0.


In [4]:
from collections import Counter

csv_path = BATCH_OUTPUT_DIR / "batch_detection_results.csv"
csv_columns = [
    "binary_name",
    "source_name",
    "source_path",
    "batch_number",
    "row_index_in_batch",
    "actual_class",
    "predicted_class",
    "predicted_class_name",
    "malware_probability",
    "shap_status",
    "top_shap_features_json",
    "llm_status",
    "llm_tier",
    "llm_confidence_score",
    "llm_explanation",
    "llm_input_tokens",
    "llm_output_tokens",
    "llm_token_usage_source",
    "error",
]

with csv_path.open("w", newline="", encoding="utf-8-sig") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=csv_columns)
    writer.writeheader()
    for report in batch_reports:
        sample = report["sample"]
        prediction = report["random_forest"]
        shap_report = report["shap"]
        llm_report = report["llm"]
        analysis = llm_report.get("analysis", {})
        usage = llm_report.get("token_usage", {})
        writer.writerow(
            {
                "binary_name": sample["binary_name"],
                "source_name": sample["source_name"],
                "source_path": sample["source_path"],
                "batch_number": sample["batch_number"],
                "row_index_in_batch": sample["row_index_in_batch"],
                "actual_class": sample["actual_class"],
                "predicted_class": prediction["predicted_class"],
                "predicted_class_name": prediction["predicted_class_name"],
                "malware_probability": prediction["malware_probability"],
                "shap_status": shap_report["status"],
                "top_shap_features_json": json.dumps(
                    shap_report["top_features"], ensure_ascii=False
                ),
                "llm_status": llm_report["status"],
                "llm_tier": analysis.get("tier"),
                "llm_confidence_score": analysis.get("confidence_score"),
                "llm_explanation": analysis.get("explanation"),
                "llm_input_tokens": usage.get("input_tokens"),
                "llm_output_tokens": usage.get("output_tokens"),
                "llm_token_usage_source": usage.get("source"),
                "error": llm_report.get("error") or shap_report.get("error"),
            }
        )

actual_counts = Counter(r["sample"]["actual_label"] for r in batch_reports)
predicted_counts = Counter(r["random_forest"]["predicted_class"] for r in batch_reports)
correct_predictions = sum(
    r["sample"]["actual_label"] == r["random_forest"]["predicted_class"]
    for r in batch_reports
)
true_positives = sum(
    r["sample"]["actual_label"] == 1
    and r["random_forest"]["predicted_class"] == 1
    for r in batch_reports
)
false_positives = sum(
    r["sample"]["actual_label"] == 0
    and r["random_forest"]["predicted_class"] == 1
    for r in batch_reports
)
false_negatives = sum(
    r["sample"]["actual_label"] == 1
    and r["random_forest"]["predicted_class"] == 0
    for r in batch_reports
)
summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "test_data_directory": TEST_DATA_DIR.relative_to(PROJECT_ROOT).as_posix(),
    "ngram_size": NGRAM_SIZE,
    "classification_rule": "Random Forest model prediction; no probability threshold override",
    "counts": {
        "test_samples": len(batch_reports),
        "actual_benign": actual_counts[0],
        "actual_cryptojacking": actual_counts[1],
        "predicted_benign": predicted_counts[0],
        "predicted_cryptojacking": predicted_counts[1],
        "llm_completed": sum(r["llm"]["status"] == "completed" for r in batch_reports),
        "llm_failed": sum(r["llm"]["status"] == "failed" for r in batch_reports),
        "llm_skipped": sum(r["llm"]["status"] == "skipped" for r in batch_reports),
    },
    "metrics": {
        "accuracy": correct_predictions / len(batch_reports) if batch_reports else None,
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
    },
    "outputs": {
        "per_sample_reports": BATCH_OUTPUT_DIR.relative_to(PROJECT_ROOT).as_posix(),
        "results_csv": csv_path.relative_to(PROJECT_ROOT).as_posix(),
        "llm_results": LLM_RESULTS_DIR.relative_to(PROJECT_ROOT).as_posix(),
    },
}
summary_path = BATCH_OUTPUT_DIR / "batch_detection_summary.json"
with summary_path.open("w", encoding="utf-8") as summary_file:
    json.dump(summary, summary_file, indent=2, ensure_ascii=False, allow_nan=False)

print(f"Per-sample reports: {BATCH_OUTPUT_DIR}")
print(f"Aggregate CSV: {csv_path}")
print(f"Batch summary: {summary_path}")
print(f"Accuracy: {summary['metrics']['accuracy']}")

NameError: name 'batch_reports' is not defined

In [ ]:
# Exact token counts reported by the LLM backend; unknown counts are not estimated.
completed_llm_reports = [
    report for report in batch_reports
    if report["llm"]["status"] == "completed"
]
token_usage_records = [
    report["llm"].get("token_usage", {})
    for report in completed_llm_reports
]
known_input_tokens = [
    usage["input_tokens"]
    for usage in token_usage_records
    if isinstance(usage.get("input_tokens"), int)
    and not isinstance(usage.get("input_tokens"), bool)
]
known_output_tokens = [
    usage["output_tokens"]
    for usage in token_usage_records
    if isinstance(usage.get("output_tokens"), int)
    and not isinstance(usage.get("output_tokens"), bool)
]
complete_usage_calls = sum(
    isinstance(usage.get("input_tokens"), int)
    and not isinstance(usage.get("input_tokens"), bool)
    and isinstance(usage.get("output_tokens"), int)
    and not isinstance(usage.get("output_tokens"), bool)
    for usage in token_usage_records
)
unknown_usage_calls = len(completed_llm_reports) - complete_usage_calls
total_input_tokens = sum(known_input_tokens)
total_output_tokens = sum(known_output_tokens)
summary["token_usage"] = {
    "completed_llm_calls": len(completed_llm_reports),
    "calls_with_both_provider_counts": complete_usage_calls,
    "calls_with_missing_counts": unknown_usage_calls,
    "total_input_tokens_reported": total_input_tokens,
    "total_output_tokens_reported": total_output_tokens,
    "source": "provider-reported counts; missing values are not estimated",
}
with summary_path.open("w", encoding="utf-8") as summary_file:
    json.dump(summary, summary_file, indent=2, ensure_ascii=False, allow_nan=False)


print("LLM token usage across the batch")
print(f"Completed LLM calls: {len(completed_llm_reports)}")
print(f"Calls with both provider counts reported: {complete_usage_calls}")
print(f"Calls with missing token counts: {unknown_usage_calls}")
print(f"Total input tokens reported: {total_input_tokens:,}")
print(f"Total output tokens reported: {total_output_tokens:,}")
if unknown_usage_calls:
    print("Totals include only provider-reported counts; missing values were not estimated.")